# Déploiement du bloc LambdaIngestor & Règle IoT Core
Ce notebook déploie la chaîne d'ingestion Serverless entre **AWS IoT Core** et **Amazon Timestream** :
1. Chargement de la configuration Timestream (`../DB/db_config.json`).
2. Création du package ZIP de `lambda_function.py` et déploiement de la fonction Lambda `SdatawayLambdaIngestor`.
3. Configuration du groupe de logs **Amazon CloudWatch** avec une rétention de **365 jours (1 an)**.
4. Création de la règle **AWS IoT Core** (`SdatawayIngestRule`) écoutant sur `coffee-GRP10/+/batch` et autorisant IoT Core à déclencher la Lambda.
5. Test de bout en bout (Invocation directe + Publication MQTT).
6. Zone de nettoyage (Teardown).

In [ ]:
import os
import json
import time
import zipfile
import boto3
from botocore.exceptions import ClientError

# 1. Lecture de la configuration de la DB générée à l'étape précédente
DB_CONFIG_PATH = "../DB/db_config.json"
if not os.path.exists(DB_CONFIG_PATH):
    raise FileNotFoundError(f"Le fichier {DB_CONFIG_PATH} est introuvable. Exécutez d'abord le notebook de la DB !")

with open(DB_CONFIG_PATH, "r") as f:
    db_config = json.load(f)

REGION = db_config["region"]
DB_NAME = db_config["database_name"]
TABLE_TELEMETRY = db_config["table_telemetry_name"]

# 2. Constantes du bloc LambdaIngestor
LAMBDA_NAME = "GRP10-SdatawayLambdaIngestor"
IOT_RULE_NAME = "SdatawayIngestRule"
MQTT_TOPIC_FILTER = "coffee-GRP10/+/batch"
LOG_RETENTION_DAYS = 365  # 1 an (Cahier des charges)
ZIP_FILENAME = "lambda_package.zip"

# 3. Initialisation des clients AWS
session = boto3.Session(region_name=REGION)
lambda_client = session.client("lambda")
iot_client = session.client("iot")
iot_data_client = session.client("iot-data")
logs_client = session.client("logs")
iam_client = session.client("iam")
sts_client = session.client("sts")
query_client = session.client("timestream-query")

ACCOUNT_ID = sts_client.get_caller_identity()["Account"]
print(f"Configuration chargée (Compte AWS : {ACCOUNT_ID}, Région : {REGION})")
print(f"Cible Timestream : {DB_NAME}.{TABLE_TELEMETRY}")

✅ Configuration chargée (Compte AWS : 352909266144, Région : us-east-1)
   Cible Timestream : SdatawayIoTDB.telemetry


In [ ]:
def get_lambda_role_arn():
    """
    Récupère automatiquement le rôle 'LabRole' fourni par AWS Academy Learner Lab.
    Si on est sur un compte AWS standard, cherche ou crée un rôle dédié.
    """
    try:
        res = iam_client.get_role(RoleName="LabRole")
        role_arn = res["Role"]["Arn"]
        print(f"Rôle AWS Academy détecté et utilisé : {role_arn}")
        return role_arn
    except ClientError:
        print("'LabRole' non trouvé, recherche/création d'un rôle IAM standard...")
        role_name = "G10-SdatawayLambdaIngestorRole"
        assume_role_policy = {
            "Version": "2012-10-17",
            "Statement": [{
                "Effect": "Allow",
                "Principal": {"Service": "lambda.amazonaws.com"},
                "Action": "sts:AssumeRole"
            }]
        }
        try:
            res = iam_client.create_role(
                RoleName=role_name,
                AssumeRolePolicyDocument=json.dumps(assume_role_policy)
            )
            role_arn = res["Role"]["Arn"]
            iam_client.attach_role_policy(
                RoleName=role_name,
                PolicyArn="arn:aws:iam::aws:policy/AmazonTimestreamFullAccess"
            )
            iam_client.attach_role_policy(
                RoleName=role_name,
                PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
            )
            print("Attente de la propagation IAM (10s)...")
            time.sleep(10)
            return role_arn
        except ClientError as e:
            if e.response["Error"]["Code"] == "EntityAlreadyExists":
                return iam_client.get_role(RoleName=role_name)["Role"]["Arn"]
            raise e

role_arn = get_lambda_role_arn()

ℹ️ 'LabRole' non trouvé, recherche/création d'un rôle IAM standard...


In [ ]:
# 1. Création de l'archive ZIP contenant lambda_function.py
if not os.path.exists("lambda_function.py"):
    raise FileNotFoundError("Le fichier 'lambda_function.py' doit être présent dans le dossier !")

with zipfile.ZipFile(ZIP_FILENAME, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("lambda_function.py")

with open(ZIP_FILENAME, "rb") as f:
    zip_bytes = f.read()

# 2. Création ou mise à jour de la fonction Lambda
env_vars = {
    "Variables": {
        "TIMESTREAM_DB_NAME": DB_NAME,
        "TIMESTREAM_TABLE_NAME": TABLE_TELEMETRY
    }
}

try:
    response = lambda_client.create_function(
        FunctionName=LAMBDA_NAME,
        Runtime="python3.12",
        Role=role_arn,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_bytes},
        Description="Ingestion légère des lots MQTT vers Amazon Timestream",
        Timeout=15,
        MemorySize=128,
        Environment=env_vars
    )
    lambda_arn = response["FunctionArn"]
    print(f"Fonction Lambda '{LAMBDA_NAME}' créée avec succès.")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceConflictException":
        print(f"La Lambda '{LAMBDA_NAME}' existe déjà, mise à jour du code et de la config...")
        lambda_client.update_function_code(FunctionName=LAMBDA_NAME, ZipFile=zip_bytes)
        waiter = lambda_client.get_waiter("function_updated")
        waiter.wait(FunctionName=LAMBDA_NAME)
        response = lambda_client.update_function_configuration(
            FunctionName=LAMBDA_NAME,
            Role=role_arn,
            Timeout=15,
            MemorySize=128,
            Environment=env_vars
        )
        lambda_arn = response["FunctionArn"]
        print(f"Fonction Lambda '{LAMBDA_NAME}' mise à jour.")
    else:
        raise e

print(f"ARN : {lambda_arn}")

✅ Fonction Lambda 'GRP10-SdatawayLambdaIngestor' créée avec succès.
   ARN : arn:aws:lambda:us-east-1:352909266144:function:GRP10-SdatawayLambdaIngestor


In [ ]:
log_group_name = f"/aws/lambda/{LAMBDA_NAME}"

try:
    logs_client.create_log_group(logGroupName=log_group_name)
    print(f"Groupe de logs CloudWatch '{log_group_name}' créé.")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceAlreadyExistsException":
        print(f"Le groupe de logs '{log_group_name}' existe déjà.")
    else:
        raise e

# Application de la rétention de 365 jours (1 an) exigée par le cahier des charges
logs_client.put_retention_policy(
    logGroupName=log_group_name,
    retentionInDays=LOG_RETENTION_DAYS
)
print(f"Politique de rétention CloudWatch configurée à {LOG_RETENTION_DAYS} jours (1 an).")

✅ Groupe de logs CloudWatch '/aws/lambda/GRP10-SdatawayLambdaIngestor' créé.
✅ Politique de rétention CloudWatch configurée à 365 jours (1 an).


In [ ]:
rule_arn = f"arn:aws:iot:{REGION}:{ACCOUNT_ID}:rule/{IOT_RULE_NAME}"

# 1. Création ou remplacement de la règle de routage IoT Core
rule_payload = {
    "sql": f"SELECT * FROM '{MQTT_TOPIC_FILTER}'",
    "description": "Route les batchs des machines à café vers LambdaIngestor",
    "actions": [
        {
            "lambda": {
                "functionArn": lambda_arn
            }
        }
    ],
    "ruleDisabled": False,
    "awsIotSqlVersion": "2016-03-23"
}

try:
    iot_client.create_topic_rule(ruleName=IOT_RULE_NAME, topicRulePayload=rule_payload)
    print(f"Règle IoT Core '{IOT_RULE_NAME}' créée sur le topic '{MQTT_TOPIC_FILTER}'.")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceAlreadyExistsException":
        iot_client.replace_topic_rule(ruleName=IOT_RULE_NAME, topicRulePayload=rule_payload)
        print(f"Règle IoT Core '{IOT_RULE_NAME}' existante mise à jour.")
    else:
        raise e

# 2. Autoriser le service AWS IoT Core à invoquer notre fonction Lambda
try:
    lambda_client.add_permission(
        FunctionName=LAMBDA_NAME,
        StatementId="AllowIoTCoreInvoke",
        Action="lambda:InvokeFunction",
        Principal="iot.amazonaws.com",
        SourceArn=rule_arn
    )
    print("Permission 'AllowIoTCoreInvoke' accordée à AWS IoT Core.")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceConflictException":
        print("La permission 'AllowIoTCoreInvoke' est déjà configurée.")
    else:
        raise e

# 3. Sauvegarde de la configuration dans lambda_config.json
lambda_config = {
    "region": REGION,
    "lambda_name": LAMBDA_NAME,
    "lambda_arn": lambda_arn,
    "iot_rule_name": IOT_RULE_NAME,
    "iot_rule_arn": rule_arn,
    "mqtt_topic_filter": MQTT_TOPIC_FILTER,
    "log_group_name": log_group_name
}
with open("lambda_config.json", "w") as f:
    json.dump(lambda_config, f, indent=2)
print("Fichier 'lambda_config.json' sauvegardé.")

✅ Règle IoT Core 'SdatawayIngestRule' créée sur le topic 'coffee-GRP10/+/batch'.
✅ Permission 'AllowIoTCoreInvoke' accordée à AWS IoT Core.
✅ Fichier 'lambda_config.json' sauvegardé.


In [ ]:
CONFIRM_TEST = False  # Passez à True pour tester l'ingestion avec un batch de mesures simulées

if CONFIRM_TEST:
    # Génération de timestamps actuels (espacés de 2s comme le simulateur) pour que Timestream les accepte
    now_ms = int(time.time() * 1000)

    test_batch = {
        "edge_id": "edge-nathan-coudray",
        "device_id": "coffee-device-30",
        "sent_at": now_ms,
        "readings": [
            {
                "timestamp": now_ms - (8 - i) * 2000,
                "device_id": "coffee-device-30",
                "uptime_ms": 822000 + i * 2000,
                "temperature_c": round(93.70 + (i * 0.05), 2),
                "power_consumption_mw": 2568.0,
                "heap_free_bytes": 50157 - (i * 10),
                "device_status": "standby",
                "error_code": 0
            }
            for i in range(9)
        ]
    }

    # Publication sur le topic MQTT réel via le client IoT Data d'AWS
    test_topic = "coffee-GRP10/coffee-device-30/batch"
    iot_data_client.publish(
        topic=test_topic,
        qos=1,
        payload=json.dumps(test_batch)
    )
    print(f"Batch de 9 mesures publié sur le topic MQTT '{test_topic}'. Attente de l'ingestion (3s)...")
    time.sleep(3)

    # Vérification dans Amazon Timestream
    query = f"""
    SELECT time, device_id, edge_id, uptime_ms, temperature_c, power_consumption_mw, heap_free_bytes, device_status, error_code
    FROM "{DB_NAME}"."{TABLE_TELEMETRY}"
    WHERE device_id = 'coffee-device-30' AND time > ago(5m)
    ORDER BY time DESC
    LIMIT 10
    """

    res = query_client.query(QueryString=query)
    cols = [c["Name"] for c in res["ColumnInfo"]]
    print(f"\n{len(res['Rows'])} lignes retrouvées dans Timestream pour 'coffee-device-30' :")
    for row in res["Rows"]:
        vals = [d.get("ScalarValue", "NULL") for d in row["Data"]]
        print(dict(zip(cols, vals)))

---
## 🧹 Zone de Nettoyage (Teardown)
Exécutez la cellule suivante pour supprimer la règle IoT Core, la fonction Lambda, le groupe de logs CloudWatch et les fichiers temporaires.

In [ ]:
CONFIRM_DELETE = False  # Passez à True pour supprimer les ressources

if CONFIRM_DELETE:
    # 1. Suppression de la règle IoT Core
    try:
        iot_client.delete_topic_rule(ruleName=IOT_RULE_NAME)
        print(f"Règle IoT Core '{IOT_RULE_NAME}' supprimée.")
    except ClientError as e:
        print(f"Erreur suppression règle IoT : {e.response['Error']['Message']}")

    # 2. Suppression de la fonction Lambda
    try:
        lambda_client.delete_function(FunctionName=LAMBDA_NAME)
        print(f"Fonction Lambda '{LAMBDA_NAME}' supprimée.")
    except ClientError as e:
        print(f"Erreur suppression Lambda : {e.response['Error']['Message']}")

    # 3. Suppression du groupe de logs CloudWatch
    try:
        logs_client.delete_log_group(logGroupName=f"/aws/lambda/{LAMBDA_NAME}")
        print(f"Groupe de logs CloudWatch supprimé.")
    except ClientError as e:
        print(f"Erreur suppression Logs : {e.response['Error']['Message']}")

    # 4. Suppression du zip local
    if os.path.exists(ZIP_FILENAME):
        os.remove(ZIP_FILENAME)
        print(f"Archive locale '{ZIP_FILENAME}' supprimée.")
else:
    print("Suppression désactivée. Passez CONFIRM_DELETE = True pour nettoyer.")

🔒 Suppression désactivée. Passez CONFIRM_DELETE = True pour nettoyer.
